In [40]:
import pandas as pd

In [41]:
df = pd.read_csv("../data/fact_team_games.csv")
df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])

In [42]:
sched = df[["GAME_ID", "GAME_DATE", "SEASON_ID", "TEAM_ID", "MATCHUP"]].copy()

In [43]:
sched["IS_HOME"] = sched["MATCHUP"].str.contains("vs.").astype(int)

In [53]:
pair = sched[["GAME_ID", "TEAM_ID"]].drop_duplicates(subset=["GAME_ID", "TEAM_ID"])

opp = (
    pair.merge(pair, on="GAME_ID", suffixes=("", "_OPP"))
        .query("TEAM_ID != TEAM_ID_OPP")
        .rename(columns={"TEAM_ID_OPP": "OPP_TEAM_ID"})
        [["GAME_ID", "TEAM_ID", "OPP_TEAM_ID"]]
        .drop_duplicates(subset=["GAME_ID", "TEAM_ID"])
)

sched = sched.merge(opp, on=["GAME_ID", "TEAM_ID"], how="left", validate="one_to_one")

In [54]:
sched["PREV_GAME_DATE"] = sched.groupby("TEAM_ID")["GAME_DATE"].shift(-1)
sched["DAYS_REST"] = (sched["GAME_DATE"] - sched["PREV_GAME_DATE"]).dt.days.fillna(0).astype("Int64")
sched["IS_BACK_TO_BACK"] = (sched["DAYS_REST"] == 1).astype(int)

In [57]:
sched["NEXT_GAME_DATE"] = sched.groupby("TEAM_ID")["GAME_DATE"].shift(1)

In [58]:
dim_schedule = sched[
    [
        "GAME_ID", "GAME_DATE", "SEASON_ID", "TEAM_ID", "OPP_TEAM_ID",
        "IS_HOME", "PREV_GAME_DATE", "DAYS_REST", "IS_BACK_TO_BACK", "NEXT_GAME_DATE"
    ]
].copy()

dim_schedule.to_csv("../data/dim_schedule.csv", index=False)